# Step 4 — Evaluation Protocol (Time Series)

In this step, we define a **time-series evaluation protocol** that prevents leakage and reflects the real forecasting workflow.

We implement:

- A **temporal split** (Train / Validation / Test)
- **Expanding window backtesting** (forecast origin moves forward, training set grows)
- Two forecasting horizons:
  - **H = 1 day ahead**
  - **H = 7 days ahead**

The output of this notebook will be reusable score tables to compare baselines (Step 5) and forecasting models.


In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
df = pd.read_csv("data_tmp/bakery_sales_top12_cleaned.csv", parse_dates=["date"])

df

,date,time,ticket_number,article,quantity,unit_price,year,month,day_of_week,hour,is_weekend
0,2021-01-02 08:38:00,08:38,150040.0,BAGUETTE,1.0,0.9,2021,1,Saturday,8,0
1,2021-01-02 08:38:00,08:38,150040.0,PAIN AU CHOCOLAT,3.0,1.2,2021,1,Saturday,8,0
2,2021-01-02 09:14:00,09:14,150041.0,PAIN AU CHOCOLAT,2.0,1.2,2021,1,Saturday,9,0
3,2021-01-02 09:25:00,09:25,150042.0,TRADITIONAL BAGUETTE,5.0,1.2,2021,1,Saturday,9,0
4,2021-01-02 09:25:00,09:25,150043.0,BAGUETTE,2.0,0.9,2021,1,Saturday,9,0
...,...,...,...,...,...,...,...,...,...,...,...
143697,2022-09-30 18:39:00,18:39,288910.0,TRADITIONAL BAGUETTE,1.0,1.3,2022,9,Friday,18,0
143698,2022-09-30 18:52:00,18:52,288911.0,CAMPAGNE,2.0,2.0,2022,9,Friday,18,0
143699,2022-09-30 18:52:00,18:52,288911.0,TRADITIONAL BAGUETTE,5.0,1.3,2022,9,Friday,18,0
143700,2022-09-30 18:55:00,18:55,288912.0,TRADITIONAL BAGUETTE,1.0,1.3,2022,9,Friday,18,0


## 3 Temporal split (Train / Validation / Test)

We use the following split:

- **Train:** 2021-01-01 → 2022-03-31  
- **Validation:** 2022-04-01 → 2022-06-30  
- **Test:** 2022-07-01 → 2022-09-30


In [5]:
def time_split(df_ts, train_end, val_end):
    train_end = pd.to_datetime(train_end)
    val_end = pd.to_datetime(val_end)

    train = df_ts.loc[:train_end]
    val = df_ts.loc[train_end + pd.Timedelta(days=1):val_end]
    test = df_ts.loc[val_end + pd.Timedelta(days=1):]
    return train, val, test

TRAIN_END = "2022-03-31"
VAL_END   = "2022-06-30"


qty_ts = pd.read_csv("data_tmp/qty_ts.csv", index_col=0, parse_dates=True)
train_ts, val_ts, test_ts = time_split(qty_ts, TRAIN_END, VAL_END)

print("Train:", train_ts.index.min().date(), "->", train_ts.index.max().date(), "| n =", len(train_ts))
print("Val:  ", val_ts.index.min().date(), "->", val_ts.index.max().date(), "| n =", len(val_ts))
print("Test: ", test_ts.index.min().date(), "->", test_ts.index.max().date(), "| n =", len(test_ts))


Train: 2021-01-02 -> 2022-03-31 | n = 420
Val:   2022-04-01 -> 2022-06-30 | n = 90
Test:  2022-07-01 -> 2022-09-30 | n = 90


## 4) Expanding window backtesting (H = 1 and H = 7)

Protocol:
- At each origin date `t`:
  - Train on all data up to `t` (expanding window)
  - Forecast the next `H` day(s)
- Move `t` forward by 1 day (`step = 1`) and repeat

We will backtest on:
- validation window (for model selection)
- test window (final evaluation)


In [ ]:
def make_backtest_folds(index, start_train_end, horizon=7, step=1):
    start_train_end = pd.to_datetime(start_train_end)
    all_dates = pd.to_datetime(index)

    train_end = start_train_end
    last_date = all_dates.max()

    while True:
        pred_start = train_end + pd.Timedelta(days=1)
        pred_end = pred_start + pd.Timedelta(days=horizon - 1)
        if pred_end > last_date:
            break
        yield train_end, pred_start, pred_end
        train_end = train_end + pd.Timedelta(days=step)


### Forecast function interface

Any model/baseline must implement:

`forecast_fn(train_df, horizon) -> pred_df`

Return:
- DataFrame with length = `horizon`
- same columns as `qty_ts`
- index = the next future dates


In [ ]:
def forecast_dummy_zero(train_df, horizon):
    last_date = train_df.index.max()
    future_index = pd.date_range(last_date + pd.Timedelta(days=1), periods=horizon, freq="D")
    return pd.DataFrame(0.0, index=future_index, columns=train_df.columns)


In [ ]:
def backtest_expanding(full_df, forecast_fn, start_train_end, end_date, horizon=7, step=1):
    end_date = pd.to_datetime(end_date)

    rows = []
    for train_end, pred_start, pred_end in make_backtest_folds(full_df.index, start_train_end, horizon=horizon, step=step):
        if pred_end > end_date:
            break

        train_df = full_df.loc[:train_end]
        y_true = full_df.loc[pred_start:pred_end]
        y_pred = forecast_fn(train_df, horizon=horizon)

        # Defensive alignment
        y_pred = y_pred.reindex(index=y_true.index, columns=full_df.columns)

        for product in full_df.columns:
            yt = y_true[product].values
            yp = y_pred[product].values

            rows.append({
                "train_end": train_end,
                "pred_start": pred_start,
                "pred_end": pred_end,
                "horizon": horizon,
                "product": product,
                "MAE": mean_absolute_error(yt, yp),
                "RMSE": np.sqrt(mean_squared_error(yt, yp)),
            })

    return pd.DataFrame(rows)


In [ ]:
VAL_END_DATE = val_ts.index.max()

scores_val_h1 = backtest_expanding(
    full_df=qty_ts,
    forecast_fn=forecast_dummy_zero,
    start_train_end=TRAIN_END,
    end_date=VAL_END_DATE,
    horizon=1,
    step=1
)

scores_val_h7 = backtest_expanding(
    full_df=qty_ts,
    forecast_fn=forecast_dummy_zero,
    start_train_end=TRAIN_END,
    end_date=VAL_END_DATE,
    horizon=7,
    step=1
)

print("Validation H=1:", scores_val_h1.shape)
print("Validation H=7:", scores_val_h7.shape)
scores_val_h1


Validation H=1: (1092, 7)
Validation H=7: (1020, 7)


,train_end,pred_start,pred_end,horizon,product,MAE,RMSE
0,2022-03-31,2022-04-01,2022-04-01,1,BAGUETTE,29.0,29.0
1,2022-03-31,2022-04-01,2022-04-01,1,BANETTE,15.0,15.0
2,2022-03-31,2022-04-01,2022-04-01,1,BOULE 400G,3.0,3.0
3,2022-03-31,2022-04-01,2022-04-01,1,CAMPAGNE,4.0,4.0
4,2022-03-31,2022-04-01,2022-04-01,1,CEREAL BAGUETTE,8.0,8.0
...,...,...,...,...,...,...,...
1087,2022-06-29,2022-06-30,2022-06-30,1,ECLAIR,0.0,0.0
1088,2022-06-29,2022-06-30,2022-06-30,1,PAIN AU CHOCOLAT,41.0,41.0
1089,2022-06-29,2022-06-30,2022-06-30,1,SPECIAL BREAD,10.0,10.0
1090,2022-06-29,2022-06-30,2022-06-30,1,TARTELETTE,9.0,9.0


In [ ]:
def summarize_scores(scores_df):
    per_product = (scores_df
                   .groupby(["horizon", "product"])[["MAE", "RMSE"]]
                   .mean()
                   .reset_index()
                   .sort_values(["horizon", "MAE"]))
    
    global_macro = (scores_df
                    .groupby(["horizon"])[["MAE", "RMSE"]]
                    .mean()
                    .reset_index())
    return per_product, global_macro

val_scores_all = pd.concat([scores_val_h1, scores_val_h7], ignore_index=True)
val_per_product, val_global = summarize_scores(val_scores_all)

display(val_global)
display(val_per_product)


,horizon,MAE,RMSE
0,1,30.781136,30.781136
1,7,31.261765,35.142091


,horizon,product,MAE,RMSE
5,1,COOKIE,5.043956,5.043956
7,1,ECLAIR,5.340659,5.340659
3,1,CAMPAGNE,5.912088,5.912088
2,1,BOULE 400G,7.384615,7.384615
10,1,TARTELETTE,8.285714,8.285714
9,1,SPECIAL BREAD,8.593407,8.593407
4,1,CEREAL BAGUETTE,10.296703,10.296703
1,1,BANETTE,33.109890,33.109890
0,1,BAGUETTE,33.340659,33.340659
8,1,PAIN AU CHOCOLAT,39.450549,39.450549


In [ ]:
TEST_END_DATE = test_ts.index.max()

scores_test_h1 = backtest_expanding(
    full_df=qty_ts,
    forecast_fn=forecast_dummy_zero,
    start_train_end=VAL_END,
    end_date=TEST_END_DATE,
    horizon=1,
    step=1
)

scores_test_h7 = backtest_expanding(
    full_df=qty_ts,
    forecast_fn=forecast_dummy_zero,
    start_train_end=VAL_END,
    end_date=TEST_END_DATE,
    horizon=7,
    step=1
)

test_scores_all = pd.concat([scores_test_h1, scores_test_h7], ignore_index=True)
test_per_product, test_global = summarize_scores(test_scores_all)

display(test_global)
display(test_per_product)


,horizon,MAE,RMSE
0,1,49.380435,49.380435
1,7,50.653516,53.706469


,horizon,product,MAE,RMSE
7,1,ECLAIR,6.586957,6.586957
5,1,COOKIE,9.000000,9.000000
3,1,CAMPAGNE,9.369565,9.369565
10,1,TARTELETTE,9.532609,9.532609
2,1,BOULE 400G,10.630435,10.630435
9,1,SPECIAL BREAD,11.369565,11.369565
4,1,CEREAL BAGUETTE,13.456522,13.456522
0,1,BAGUETTE,44.586957,44.586957
1,1,BANETTE,50.282609,50.282609
8,1,PAIN AU CHOCOLAT,60.728261,60.728261


### Interpretation

The expanding-window backtesting highlights strong non-stationarity in demand. Forecast errors increase substantially from validation to test, indicating level shifts in the final months of the dataset. Errors are highly heterogeneous across products: low-volume items exhibit small absolute errors, while high-volume staples (especially traditional baguette) dominate the global error metrics. Weekly forecasts (H=7) are consistently more difficult than one-day-ahead forecasts, reinforcing the need for models that explicitly capture weekly seasonality. These results motivate the use of seasonal baselines and product-specific modeling strategies in the next step.